# 02 — Preprocessing & Feature Engineering: Metro Interstate Traffic Volume

**Goal:** turn the raw CSV into a clean, model-ready feature matrix, applying fixes for every issue found in `01_eda.ipynb`.

**Steps, in order:**
1. Load data, parse `date_time` properly
2. Drop exact duplicate rows
3. Handle rows sharing a timestamp with another row
4. Fix `temp` = 0 Kelvin (invalid readings)
5. Fix `rain_1h` outlier (invalid reading)
6. Engineer `is_holiday` binary flag
7. Extract `hour`, `day_of_week`, `month` from `date_time`
8. One-hot encode `weather_main`; drop `weather_description`
9. Save the cleaned feature matrix to `data/processed/`


## Step 1: load data, parse `date_time` properly

Learned from Stage 3's mistake: parse `date_time` into a real datetime type immediately, before doing anything else with it.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/Metro_Interstate_Traffic_Volume.csv')
df['date_time'] = pd.to_datetime(df['date_time'], format='%d-%m-%Y %H:%M')

df.shape, df['date_time'].min(), df['date_time'].max()


((48204, 9),
 Timestamp('2012-10-02 09:00:00'),
 Timestamp('2018-09-30 23:00:00'))

## Step 2: drop exact duplicate rows

Found in EDA: 17 fully identical rows. These carry no extra information — safe to drop outright.

In [2]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

print('Rows before:', before)
print('Rows after:', after)
print('Dropped:', before - after)


Rows before: 48204
Rows after: 48187
Dropped: 17


## Step 3: handle rows sharing a timestamp

Found in EDA: many rows share a `date_time` with another row (likely multiple weather conditions logged in the same hour). Before deciding how to handle this, check something important first: **does `traffic_volume` itself ever differ between rows that share an hour?** If it doesn't, keeping the first row per hour loses nothing about the target — only possibly-redundant weather detail. If it does differ, that's a bigger problem worth investigating further.

In [3]:
dup_hour_mask = df.duplicated(subset='date_time', keep=False)
dup_hours = df[dup_hour_mask]

traffic_variation = dup_hours.groupby('date_time')['traffic_volume'].nunique()
print('Rows sharing an hour with another row:', dup_hour_mask.sum())
print('Of those hours, how many have traffic_volume that DISAGREES within the group:')
print((traffic_variation > 1).sum(), 'out of', traffic_variation.shape[0], 'duplicated hours')


Rows sharing an hour with another row: 13042
Of those hours, how many have traffic_volume that DISAGREES within the group:
0 out of 5430 duplicated hours


In [4]:
before = len(df)
df = df.sort_values('date_time').drop_duplicates(subset='date_time', keep='first')
after = len(df)

print('Rows before:', before)
print('Rows after:', after)
print('Dropped:', before - after)


Rows before: 48187
Rows after: 40575
Dropped: 7612


## Step 4: fix `temp` = 0 Kelvin (invalid readings)

Found in EDA: 0 Kelvin is physically impossible — these are sensor errors, not real readings. Treat them as missing, then impute using nearby real readings (temperature changes gradually hour-to-hour, so a neighboring hour's value is a reasonable estimate) rather than dropping the rows outright, since we'd lose the `traffic_volume` reading too.

In [5]:
invalid_temp_count = (df['temp'] == 0).sum()
print('Invalid temp=0 readings found:', invalid_temp_count)

df['temp'] = df['temp'].replace(0, np.nan)
df = df.sort_values('date_time')
df['temp'] = df['temp'].interpolate(method='linear', limit_direction='both')

print('Remaining nulls in temp after interpolation:', df['temp'].isna().sum())
print('New temp min:', df['temp'].min())


Invalid temp=0 readings found: 10
Remaining nulls in temp after interpolation: 0
New temp min: 243.39


## Step 5: fix `rain_1h` outlier (invalid reading)

Found in EDA: a max of 9,831.3mm in one hour — far beyond any real-world hourly rainfall record (~300mm is the real-world record). First check how many rows are actually affected before deciding how to fix it — same principle as step 4, don't assume it's just one row without checking.

In [6]:
plausible_max_mm = 300  # real-world hourly rainfall record is ~300mm
invalid_rain_mask = df['rain_1h'] > plausible_max_mm

print('Rows with implausible rain_1h (>300mm):', invalid_rain_mask.sum())
print(df.loc[invalid_rain_mask, ['date_time', 'rain_1h']])


Rows with implausible rain_1h (>300mm): 1
                date_time  rain_1h
24872 2016-07-11 17:00:00   9831.3


In [7]:
df.loc[invalid_rain_mask, 'rain_1h'] = np.nan
df['rain_1h'] = df['rain_1h'].interpolate(method='linear', limit_direction='both')

print('Remaining nulls in rain_1h:', df['rain_1h'].isna().sum())
print('New rain_1h max:', df['rain_1h'].max())


Remaining nulls in rain_1h: 0
New rain_1h max: 55.63


## Step 6: engineer `is_holiday` binary flag

Decided back in EDA: `holiday` is 99.9% blank by design (only filled in on the actual holiday hour). Rather than one-hot encoding 11 separate holiday names, collapse it into a single `True`/`False` flag — the business question cares about "is it a holiday," not "which specific holiday."

In [8]:
df['is_holiday'] = df['holiday'].notna().astype(int)
df = df.drop(columns=['holiday'])

print(df['is_holiday'].value_counts())


is_holiday
0    40522
1       53
Name: count, dtype: int64


## Step 7: extract `hour`, `day_of_week`, `month` from `date_time`

Simple idea: `date_time` itself isn't something a model can use directly — it can't compare "2016-07-11 17:00" to another timestamp in a useful way. So we break it into simple numbers the model can actually learn from: what hour of the day (we already proved this matters a lot — day vs. night), what day of the week (weekday commute vs. weekend), and what month (season/weather patterns).

In [9]:
df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek  # 0 = Monday, 6 = Sunday
df['month'] = df['date_time'].dt.month

df[['date_time', 'hour', 'day_of_week', 'month']].head()


,date_time,hour,day_of_week,month
0,2012-10-02 09:00:00,9,1,10
1,2012-10-02 10:00:00,10,1,10
2,2012-10-02 11:00:00,11,1,10
3,2012-10-02 12:00:00,12,1,10
4,2012-10-02 13:00:00,13,1,10


## Step 8: one-hot encode `weather_main`; drop `weather_description`

Decided back in EDA: `weather_main` (11 clean categories) gets one-hot encoded — one new True/False column per weather type. `weather_description` (38 categories, had a capitalization bug, mostly repeats what `weather_main` already says) gets dropped instead of encoded.

In [10]:
weather_dummies = pd.get_dummies(df['weather_main'], prefix='weather')
df = pd.concat([df, weather_dummies], axis=1)
df = df.drop(columns=['weather_main', 'weather_description'])

df.columns.tolist()


['traffic_volume',
 'temp',
 'rain_1h',
 'snow_1h',
 'clouds_all',
 'date_time',
 'is_holiday',
 'hour',
 'day_of_week',
 'month',
 'weather_Clear',
 'weather_Clouds',
 'weather_Drizzle',
 'weather_Fog',
 'weather_Haze',
 'weather_Mist',
 'weather_Rain',
 'weather_Smoke',
 'weather_Snow',
 'weather_Squall',
 'weather_Thunderstorm']

## Step 9: save the cleaned feature matrix

Last step — write the fully cleaned, feature-engineered table to `data/processed/`. This is the file Stage 6 (modeling) will load from — not the raw CSV.

In [11]:
df.to_csv('../data/processed/traffic_features_clean.csv', index=False)

print('Saved shape:', df.shape)
print('Columns:', df.columns.tolist())


Saved shape: (40575, 21)
Columns: ['traffic_volume', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'date_time', 'is_holiday', 'hour', 'day_of_week', 'month', 'weather_Clear', 'weather_Clouds', 'weather_Drizzle', 'weather_Fog', 'weather_Haze', 'weather_Mist', 'weather_Rain', 'weather_Smoke', 'weather_Snow', 'weather_Squall', 'weather_Thunderstorm']
